# Module 4: Data Quality Framework

**Objective**: Build a reusable data quality validation framework for banking data.

## What You'll Learn
- Null/missing value detection
- Data type validation
- Business rule validation
- Anomaly detection
- Quality score calculation

In [ ]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module04-DataQuality").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
customers_df = spark.read.parquet(f"{S3_RAW}/customers") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "customers.csv"), header=True, inferSchema=True)
transactions_df = spark.read.parquet(f"{S3_RAW}/transactions") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)

print(f"Customers: {customers_df.count()}, Transactions: {transactions_df.count()}")

## 1. Null Value Analysis

In [ ]:
def null_analysis(df, df_name):
    """Analyze null values in each column."""
    total = df.count()
    print(f"\n=== Null Analysis: {df_name} ({total:,} rows) ===")
    for c in df.columns:
        null_count = df.filter(col(c).isNull() | isnan(c)).count()
        pct = (null_count / total) * 100
        if null_count > 0:
            print(f"  {c}: {null_count:,} nulls ({pct:.2f}%)")
    print("  ✅ All columns checked")

null_analysis(customers_df, "Customers")
null_analysis(transactions_df, "Transactions")

## 2. Business Rule Validation

In [ ]:
# Define validation rules
def validate_transactions(df):
    """Validate transaction business rules."""
    validations = [
        ("amount_positive", col("amount") > 0),
        ("status_valid", col("status").isin(["Completed", "Pending", "Failed", "Reversed"])),
        ("channel_valid", col("channel").isin(["Branch", "ATM", "Mobile App", "Internet Banking", "POS", "API"])),
    ]
    
    result = df
    for rule_name, rule_expr in validations:
        result = result.withColumn(rule_name, when(rule_expr, True).otherwise(False))
    
    return result

txn_validated = validate_transactions(transactions_df)
txn_validated.select("txn_id", "amount", "status", "channel", "amount_positive", "status_valid", "channel_valid").show(5)

In [ ]:
# Validation summary
total = txn_validated.count()
for rule in ["amount_positive", "status_valid", "channel_valid"]:
    passed = txn_validated.filter(col(rule)).count()
    print(f"{rule}: {passed:,}/{total:,} passed ({100*passed/total:.2f}%)")

## 3. Anomaly Detection

In [ ]:
# Statistical anomaly detection (>3 std from mean)
from pyspark.sql.functions import mean, stddev

stats = transactions_df.select(
    mean("amount").alias("mean_amt"),
    stddev("amount").alias("std_amt")
).collect()[0]

mean_amt, std_amt = stats["mean_amt"], stats["std_amt"]
threshold = mean_amt + (3 * std_amt)
print(f"Mean: {mean_amt:,.0f}, Std: {std_amt:,.0f}, Threshold (3σ): {threshold:,.0f}")

anomalies = transactions_df.filter(col("amount") > threshold)
print(f"\nAnomalous transactions (>3σ): {anomalies.count():,}")
anomalies.orderBy(col("amount").desc()).show(5)

## 4. Quality Score Calculation

In [ ]:
def calculate_quality_score(df):
    """Calculate overall data quality score (0-100)."""
    total = df.count()
    
    # Completeness: % of non-null values
    null_counts = [df.filter(col(c).isNull()).count() for c in df.columns]
    completeness = 100 * (1 - sum(null_counts) / (total * len(df.columns)))
    
    return round(completeness, 2)

score = calculate_quality_score(customers_df)
print(f"Customer Data Quality Score: {score}/100")

## Practice Exercises
1. Add email format validation for customers
2. Detect duplicate transaction IDs
3. Validate date ranges (no future dates)

In [ ]:
spark.stop()